<a href="https://colab.research.google.com/github/Rut092/rut-ai-portfolio-PHASE3-Computer-Vision/blob/main/CNN_architecture(insider_modules).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implementation of Residual Network

In [1]:
import torch
import torch.nn as nn

In [2]:
class ResidualBlock(nn.Module):
    def __init__(self,in_channels,out_channels,stride = 1):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,out_channels,stride = stride, bias = False, padding = 1, kernel_size = 3),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(out_channels,out_channels,stride = stride, bias = False, padding = 1, kernel_size = 3),
            nn.BatchNorm2d(out_channels)
        )

        self.shortcut = nn.Sequential()
        if stride!=1 or in_channels!=out_channels:
            self.shortcut = nn.Squential(
                nn.Conv2d(in_channels,out_channels,stride = stride, bias = False, padding = 1, kernel_size = 3),
            nn.BatchNorm2d(out_channels)
            )

        self.relu = nn.ReLU(inplace = True)

    def forward(self,x):

        residual = self.shortcut(x)

        out = self.block1(x)
        out = self.block2(out)

        out+=residual
        out = self.relu(out)

        return out

In [3]:
# dummy image of batch 2
dummy = torch.rand(2,64,56,56)
res_block = ResidualBlock(in_channels = 64, out_channels = 64,stride = 1)
output = res_block(dummy)

print(f"Input Shape: {dummy.shape}")
print(f"Output Shape: {output.shape}")

Input Shape: torch.Size([2, 64, 56, 56])
Output Shape: torch.Size([2, 64, 56, 56])


## Implementation of Inception Module

In [4]:
class Inception(nn.Module):
    def __init__(self, in_channels, out_1x1, red_3x3, out_3x3, red_5x5, out_5x5, out_pool):
        super().__init__()

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels,out_1x1,kernel_size = 1),
            nn.ReLU(inplace = True)

        )

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, red_3x3, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(red_3x3, out_3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, red_5x5, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(red_5x5, out_5x5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )
        # note after end result i have made them same dim using padding width and height

        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Pass input through all 4 parallel branches
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        out4 = self.branch4(x)

        return torch.cat([out1, out2, out3, out4], dim=1)



In [5]:
dummy_input = torch.randn(2, 192, 28, 28)

inception = Inception(
        in_channels=192,
        out_1x1=64,
        red_3x3=96,
        out_3x3=128,
        red_5x5=16,
        out_5x5=32,
        out_pool=32
    )
# 64 + 128 + 32 + 32 => 256
output = inception(dummy_input)
print("Input Shape: ", dummy_input.shape)
print("Output Shape:", output.shape)

Input Shape:  torch.Size([2, 192, 28, 28])
Output Shape: torch.Size([2, 256, 28, 28])
